<div align="center">
    <img src="../../../media/a365-agents.png" width="100%" alt="Microsoft Foundry workshop / lab / sample"> 
</div>

# Build a Foundry agent that is managed and governed with Agent 365

In this lab you will:

1. **Build or reuse a pro-code Foundry agent** with the `azure-ai-projects` SDK and the existing workshop tools.
2. **Resolve the Foundry resource** that hosts the agent.
3. **Verify or enable Agent 365 data collection** on the Foundry resource.
4. **Create a local governance evidence pack** that captures the Foundry agent, resource, and Agent 365 readiness state.
5. **Confirm the agent in Agent 365** in the Microsoft 365 admin center.
6. **Apply and test Agent 365 governance controls** such as visibility, availability, access, audit, and lifecycle management.
7. **Clean up local generated files** so the notebook can be re-run safely.

## Architecture

```text
 ┌─────────────────────────────┐
 │ Microsoft Foundry           │
 │ Foundry resource + project  │
 │ Foundry agent runtime       │
 └──────────────┬──────────────┘
                │
                │ registry sync + activity data collection
                ▼
 ┌──────────────────────────────────────────────────────────┐
 │ Agent 365 governance plane                              │
 │ registry · access · policies · audit · lifecycle         │
 └──────────────────────────────────────────────────────────┘
```

This lab uses the **Foundry-originated Agent 365 governance path**. The purpose is to demonstrate that agents created in Microsoft Foundry can be discovered, managed, and governed from Agent 365.

> **Run the main workshop first.** This lab assumes you completed
> [`src/workshop/README.md`](../../workshop/README.md) and have a populated
> [`src/workshop/.env`](../../workshop/.env).


## Prerequisites

Before running this notebook:

| # | Requirement | How to verify |
|---|-------------|---------------|
| 1 | Main workshop completed | `../../workshop/.env` exists |
| 2 | Azure CLI logged in | `az account show` |
| 3 | Permission to create/read Foundry agents | Azure AI User or equivalent on the Foundry project |
| 4 | Permission to read/update the Foundry Azure resource | Owner, Contributor, or role with `Microsoft.CognitiveServices/accounts/read` and update permissions |
| 5 | Agent 365 enabled in the Microsoft 365 tenant | Microsoft 365 admin center → Agents |
| 6 | Agent 365 consent and licensing completed | Tenant admin confirms Agent 365 is active |
| 7 | Access to Microsoft 365 admin center | Required for the governance validation steps |

The notebook creates and validates the Foundry-side setup. The final governance confirmation is performed in the Microsoft 365 admin center.


## Setup — install lab dependencies

Run this once in the workshop virtual environment.


In [1]:
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Step 0 — Load environment and define helpers


In [2]:
import json
import os
import sys
import time
import uuid
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse

import httpx
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Load the workshop .env.
WORKSHOP_ENV = Path("../../workshop/.env").resolve()
assert WORKSHOP_ENV.exists(), f"Run the main workshop first — {WORKSHOP_ENV} not found."
load_dotenv(WORKSHOP_ENV)

PROJECT_ENDPOINT = os.environ["PROJECT_ENDPOINT"]
MODEL_DEPLOYMENT_NAME = os.environ["AGENT_MODEL_DEPLOYMENT_NAME"]
AZURE_SUBSCRIPTION_ID = os.environ["AZURE_SUBSCRIPTION_ID"]
AZURE_RESOURCE_GROUP_NAME = os.environ["AZURE_RESOURCE_GROUP_NAME"]

# Optional overrides for this lab.
AGENT_BASE_NAME = os.environ.get("AGENT365_AGENT_BASE_NAME", "foundry-lab-agent")
APP_NAME_SHORT = os.environ.get("AGENT365_APP_NAME_SHORT", "Foundry Lab Agent")
FOUNDRY_ARM_API_VERSION = os.environ.get("FOUNDRY_ARM_API_VERSION", "2026-03-15-preview")
ENABLE_A365_LOGGING = os.environ.get("ENABLE_A365_LOGGING", "true").lower() == "true"

# Make agent_app.py importable.
sys.path.insert(0, str(Path.cwd()))
import agent_app  # noqa: E402

credential = DefaultAzureCredential()

LAB_OUTPUT_DIR = Path("agent365-governance")
LAB_OUTPUT_DIR.mkdir(exist_ok=True)

def arm_request(method: str, path: str, **kwargs) -> httpx.Response:
    """Direct Azure Resource Manager request helper."""
    token = credential.get_token("https://management.azure.com/.default").token
    headers = kwargs.pop("headers", {})
    headers = {
        **headers,
        "Authorization": f"Bearer {token}",
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
    url = path if path.startswith("https://") else f"https://management.azure.com{path}"
    return httpx.request(method, url, headers=headers, timeout=60, **kwargs)

def run_cli(args, check=False, cwd=None):
    """Run a CLI command and return CompletedProcess."""
    print("$", " ".join(str(a) for a in args))
    result = subprocess.run(
        [str(a) for a in args],
        cwd=cwd,
        capture_output=True,
        text=True,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(args)}")
    return result

def get_signed_in_user_hint() -> str:
    """Best-effort user suffix for deterministic naming."""
    env_user = (
        os.environ.get("USER_PRINCIPAL_NAME")
        or os.environ.get("AZURE_USERNAME")
        or os.environ.get("USERNAME")
        or os.environ.get("USER")
    )
    if env_user:
        return env_user.split("@")[0].replace(".", "-").replace("_", "-")

    result = run_cli(["az", "ad", "signed-in-user", "show", "--query", "userPrincipalName", "-o", "tsv"])
    if result.returncode == 0 and result.stdout.strip():
        return result.stdout.strip().split("@")[0].replace(".", "-").replace("_", "-")

    return "user"

def get_deterministic_agent_name(base_name: str = "foundry-lab-agent") -> str:
    user_hint = get_signed_in_user_hint()
    env_hash = hashlib.sha1(str(WORKSHOP_ENV).encode()).hexdigest()[:6]
    return f"{base_name}-{user_hint}-{env_hash}".lower()

def first_existing_attr(obj, attr_names, default=None):
    for attr in attr_names:
        if hasattr(obj, attr):
            value = getattr(obj, attr)
            if value is not None:
                return value
    return default

def to_jsonable(value):
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, (list, tuple)):
        return [to_jsonable(v) for v in value]
    if isinstance(value, dict):
        return {str(k): to_jsonable(v) for k, v in value.items()}
    return str(value)

AGENT_NAME = get_deterministic_agent_name(AGENT_BASE_NAME)

print("[INFO] Foundry endpoint:", PROJECT_ENDPOINT)
print("[INFO] Model deployment:", MODEL_DEPLOYMENT_NAME)
print("[INFO] Azure subscription:", AZURE_SUBSCRIPTION_ID)
print("[INFO] Azure resource group:", AZURE_RESOURCE_GROUP_NAME)
print("[INFO] Agent name:", AGENT_NAME)
print("[INFO] Lab output directory:", LAB_OUTPUT_DIR.resolve())


[INFO] Foundry endpoint: https://aif-aiagents-vbds.services.ai.azure.com/api/projects/workshop-project
[INFO] Model deployment: gpt4o
[INFO] Azure subscription: c200e3e7-0839-483b-848b-25a98451f2cd
[INFO] Azure resource group: rg-kotp-temp
[INFO] Agent name: foundry-lab-agent-vscode-dd3ae0
[INFO] Lab output directory: /workspaces/Microsoft-Foundry/src/samples/create-agent365-managed-agents/agent365-governance


## Step 1 — Resolve the Foundry and Agent 365 governance model

For this lab:

- The **Foundry agent** is created or reused inside the Foundry project.
- The **Foundry resource** controls Agent 365 data collection through its Azure Resource Manager properties.
- The **Agent 365 registry** is the admin-facing governance plane where Foundry-originated agents can be discovered and governed.
- The notebook writes a local governance evidence pack so participants can validate what was created and where to look in Agent 365.


In [3]:
mode = {
    "integrationMode": "foundry_agent365_governance",
    "purpose": "Demonstrate Foundry agent visibility and governance in Agent 365.",
}

print(json.dumps(mode, indent=2))


{
  "integrationMode": "foundry_agent365_governance",
  "purpose": "Demonstrate Foundry agent visibility and governance in Agent 365."
}


## Step 2 — Build or reuse the Foundry agent

The Foundry agent is the runtime object that will appear in the Agent 365 registry when the tenant and resource are configured for Agent 365.


In [4]:
from agent_app import build_or_get_agent

project_client = AIProjectClient(PROJECT_ENDPOINT, credential)

agent_handle = build_or_get_agent(
    project_client=project_client,
    model_deployment_name=MODEL_DEPLOYMENT_NAME,
    agent_name=AGENT_NAME,
)

FOUNDRY_AGENT_ID = first_existing_attr(
    agent_handle,
    ["id", "agent_id", "assistant_id", "name"],
    default=AGENT_NAME,
)

FOUNDRY_AGENT_NAME = first_existing_attr(agent_handle, ["name"], default=AGENT_NAME)
FOUNDRY_AGENT_VERSION = first_existing_attr(agent_handle, ["version"], default=None)
FOUNDRY_AGENT_DESCRIPTION = first_existing_attr(agent_handle, ["description"], default=None)

print("[INFO] Foundry agent name:", FOUNDRY_AGENT_NAME)
print("[INFO] Foundry agent id/name handle:", FOUNDRY_AGENT_ID)
print("[INFO] Foundry agent version:", FOUNDRY_AGENT_VERSION)
print("[INFO] Foundry agent description:", FOUNDRY_AGENT_DESCRIPTION)


[INFO] Foundry agent name: foundry-lab-agent-vscode-dd3ae0
[INFO] Foundry agent id/name handle: foundry-lab-agent-vscode-dd3ae0
[INFO] Foundry agent version: 1
[INFO] Foundry agent description: None


## Step 3 — Discover the Foundry Azure resource

Agent 365 data collection is configured at the Foundry resource level. This step resolves the Azure resource that hosts the project from the workshop endpoint and resource group.

If automatic discovery cannot find the resource, set `FOUNDRY_ACCOUNT_NAME` in `../../workshop/.env` and re-run this step.


In [5]:
def infer_project_name_from_endpoint(endpoint: str) -> str | None:
    parsed = urlparse(endpoint)
    parts = [p for p in parsed.path.split("/") if p]
    if "projects" in parts:
        index = parts.index("projects")
        if index + 1 < len(parts):
            return parts[index + 1]
    return None

def infer_account_name_from_endpoint(endpoint: str) -> str | None:
    parsed = urlparse(endpoint)
    if parsed.hostname:
        return parsed.hostname.split(".")[0]
    return None

def get_foundry_account_resource(account_name: str):
    resource_id = (
        f"/subscriptions/{AZURE_SUBSCRIPTION_ID}"
        f"/resourceGroups/{AZURE_RESOURCE_GROUP_NAME}"
        f"/providers/Microsoft.CognitiveServices/accounts/{account_name}"
    )
    response = arm_request(
        "GET",
        resource_id,
        params={"api-version": FOUNDRY_ARM_API_VERSION},
    )
    if response.status_code < 400:
        return response.json()
    return None

def list_cognitive_services_accounts():
    scope = (
        f"/subscriptions/{AZURE_SUBSCRIPTION_ID}"
        f"/resourceGroups/{AZURE_RESOURCE_GROUP_NAME}"
        f"/providers/Microsoft.CognitiveServices/accounts"
    )
    response = arm_request(
        "GET",
        scope,
        params={"api-version": FOUNDRY_ARM_API_VERSION},
    )
    if response.status_code >= 400:
        raise RuntimeError(
            "Failed to list Microsoft.CognitiveServices/accounts.\n"
            f"Status: {response.status_code}\n"
            f"Body: {response.text}"
        )
    return response.json().get("value", [])

PROJECT_NAME = infer_project_name_from_endpoint(PROJECT_ENDPOINT)
INFERRED_ACCOUNT_NAME = infer_account_name_from_endpoint(PROJECT_ENDPOINT)
FOUNDRY_ACCOUNT_NAME = os.environ.get("FOUNDRY_ACCOUNT_NAME", INFERRED_ACCOUNT_NAME)

foundry_account = get_foundry_account_resource(FOUNDRY_ACCOUNT_NAME) if FOUNDRY_ACCOUNT_NAME else None

if not foundry_account:
    accounts = list_cognitive_services_accounts()
    print("[INFO] Foundry account could not be resolved directly.")
    print("[INFO] Candidate Microsoft.CognitiveServices/accounts in the resource group:")
    for account in accounts:
        print(" -", account.get("name"), account.get("kind"), account.get("location"))

    if len(accounts) == 1:
        foundry_account = accounts[0]
        FOUNDRY_ACCOUNT_NAME = foundry_account.get("name")
        print("[INFO] Using the only candidate account:", FOUNDRY_ACCOUNT_NAME)
    else:
        raise RuntimeError(
            "Could not identify the Foundry account. Set FOUNDRY_ACCOUNT_NAME in ../../workshop/.env."
        )

FOUNDRY_ACCOUNT_RESOURCE_ID = foundry_account["id"]
FOUNDRY_ACCOUNT_LOCATION = foundry_account.get("location")
FOUNDRY_ACCOUNT_KIND = foundry_account.get("kind")

print("[INFO] Project name:", PROJECT_NAME)
print("[INFO] Foundry account name:", FOUNDRY_ACCOUNT_NAME)
print("[INFO] Foundry account kind:", FOUNDRY_ACCOUNT_KIND)
print("[INFO] Foundry account location:", FOUNDRY_ACCOUNT_LOCATION)
print("[INFO] Foundry account resource id:", FOUNDRY_ACCOUNT_RESOURCE_ID)


[INFO] Project name: workshop-project
[INFO] Foundry account name: aif-aiagents-vbds
[INFO] Foundry account kind: AIServices
[INFO] Foundry account location: swedencentral
[INFO] Foundry account resource id: /subscriptions/c200e3e7-0839-483b-848b-25a98451f2cd/resourceGroups/rg-kotp-temp/providers/Microsoft.CognitiveServices/accounts/aif-aiagents-vbds


## Step 4 — Verify or enable Agent 365 data collection on the Foundry resource

Agent 365 data collection is controlled through Foundry resource properties. This step reads the current state and, by default, enables `a365LoggingEnabled` for the Foundry resource.

Data only flows when the Microsoft 365 tenant has Agent 365 licensing and consent in place. The Azure resource setting alone is not sufficient without tenant-level Agent 365 enablement.


In [6]:
def read_foundry_account(resource_id: str):
    response = arm_request(
        "GET",
        resource_id,
        params={"api-version": FOUNDRY_ARM_API_VERSION},
    )
    if response.status_code >= 400:
        raise RuntimeError(
            "Failed to read Foundry account.\n"
            f"Status: {response.status_code}\n"
            f"URL: {response.request.url}\n"
            f"Body: {response.text}"
        )
    return response.json()

def update_a365_logging(resource_id: str, enabled: bool):
    response = arm_request(
        "PATCH",
        resource_id,
        params={"api-version": FOUNDRY_ARM_API_VERSION},
        json={"properties": {"a365LoggingEnabled": enabled}},
    )
    if response.status_code >= 400:
        raise RuntimeError(
            "Failed to update a365LoggingEnabled.\n"
            f"Status: {response.status_code}\n"
            f"URL: {response.request.url}\n"
            f"Body: {response.text}"
        )
    return response.json()

before = read_foundry_account(FOUNDRY_ACCOUNT_RESOURCE_ID)
before_props = before.get("properties", {})

print("[INFO] Current Agent 365 properties:")
print(json.dumps({
    "a365LoggingEnabled": before_props.get("a365LoggingEnabled"),
    "a365Status": before_props.get("a365Status"),
}, indent=2))

if ENABLE_A365_LOGGING and before_props.get("a365LoggingEnabled") is not True:
    print("[INFO] Enabling Agent 365 data collection on the Foundry resource.")
    update_a365_logging(FOUNDRY_ACCOUNT_RESOURCE_ID, True)
else:
    print("[INFO] No update required for a365LoggingEnabled.")

after = read_foundry_account(FOUNDRY_ACCOUNT_RESOURCE_ID)
after_props = after.get("properties", {})

A365_LOGGING_ENABLED = after_props.get("a365LoggingEnabled")
A365_STATUS = after_props.get("a365Status")

print("[INFO] Effective Agent 365 properties:")
print(json.dumps({
    "a365LoggingEnabled": A365_LOGGING_ENABLED,
    "a365Status": A365_STATUS,
}, indent=2))

assert A365_LOGGING_ENABLED is True, (
    "a365LoggingEnabled is not true. Set ENABLE_A365_LOGGING=true or enable Agent 365 data collection manually."
)

print("[SUCCESS] Foundry resource is configured for Agent 365 data collection.")


[INFO] Current Agent 365 properties:
{
  "a365LoggingEnabled": true,
  "a365Status": "Enabled"
}
[INFO] No update required for a365LoggingEnabled.
[INFO] Effective Agent 365 properties:
{
  "a365LoggingEnabled": true,
  "a365Status": "Enabled"
}
[SUCCESS] Foundry resource is configured for Agent 365 data collection.


## Step 5 — Write the Agent 365 governance evidence pack

The evidence pack captures the Foundry agent, the Foundry resource, and the Agent 365 data collection state. Use it during the workshop to explain what was created and where the governance controls apply.


In [7]:
evidence = {
    "generatedAtUtc": datetime.now(timezone.utc).isoformat(),
    "integrationMode": "foundry_agent365_governance",
    "azure": {
        "subscriptionId": AZURE_SUBSCRIPTION_ID,
        "resourceGroup": AZURE_RESOURCE_GROUP_NAME,
    },
    "foundry": {
        "projectEndpoint": PROJECT_ENDPOINT,
        "projectName": PROJECT_NAME,
        "accountName": FOUNDRY_ACCOUNT_NAME,
        "accountKind": FOUNDRY_ACCOUNT_KIND,
        "accountLocation": FOUNDRY_ACCOUNT_LOCATION,
        "accountResourceId": FOUNDRY_ACCOUNT_RESOURCE_ID,
        "modelDeploymentName": MODEL_DEPLOYMENT_NAME,
        "agentName": FOUNDRY_AGENT_NAME,
        "agentIdOrHandle": to_jsonable(FOUNDRY_AGENT_ID),
        "agentVersion": to_jsonable(FOUNDRY_AGENT_VERSION),
        "agentDescription": to_jsonable(FOUNDRY_AGENT_DESCRIPTION),
    },
    "agent365": {
        "a365LoggingEnabled": A365_LOGGING_ENABLED,
        "a365Status": to_jsonable(A365_STATUS),
        "adminCenterUrl": "https://admin.microsoft.com",
        "expectedRegistryName": FOUNDRY_AGENT_NAME,
    },
}

evidence_json_path = LAB_OUTPUT_DIR / "agent365-governance-evidence.json"
evidence_md_path = LAB_OUTPUT_DIR / "agent365-governance-evidence.md"

with open(evidence_json_path, "w", encoding="utf-8") as f:
    json.dump(evidence, f, indent=2)

markdown = f"""# Agent 365 governance evidence

Generated at: `{evidence['generatedAtUtc']}`

## Foundry agent

| Field | Value |
|---|---|
| Agent name | `{FOUNDRY_AGENT_NAME}` |
| Agent id or handle | `{FOUNDRY_AGENT_ID}` |
| Agent version | `{FOUNDRY_AGENT_VERSION}` |
| Model deployment | `{MODEL_DEPLOYMENT_NAME}` |
| Project endpoint | `{PROJECT_ENDPOINT}` |

## Foundry resource

| Field | Value |
|---|---|
| Account name | `{FOUNDRY_ACCOUNT_NAME}` |
| Resource group | `{AZURE_RESOURCE_GROUP_NAME}` |
| Location | `{FOUNDRY_ACCOUNT_LOCATION}` |
| Kind | `{FOUNDRY_ACCOUNT_KIND}` |
| Resource id | `{FOUNDRY_ACCOUNT_RESOURCE_ID}` |

## Agent 365 readiness

| Field | Value |
|---|---|
| Integration mode | `foundry_agent365_governance` |
| a365LoggingEnabled | `{A365_LOGGING_ENABLED}` |
| a365Status | `{A365_STATUS}` |

## Admin center validation

Open `https://admin.microsoft.com`, go to **Agents**, and search for `{FOUNDRY_AGENT_NAME}`.
"""

with open(evidence_md_path, "w", encoding="utf-8") as f:
    f.write(markdown)

print("[INFO] Evidence JSON:", evidence_json_path.resolve())
print("[INFO] Evidence Markdown:", evidence_md_path.resolve())
print(json.dumps(evidence, indent=2))


[INFO] Evidence JSON: /workspaces/Microsoft-Foundry/src/samples/create-agent365-managed-agents/agent365-governance/agent365-governance-evidence.json
[INFO] Evidence Markdown: /workspaces/Microsoft-Foundry/src/samples/create-agent365-managed-agents/agent365-governance/agent365-governance-evidence.md
{
  "generatedAtUtc": "2026-05-13T11:24:28.913909+00:00",
  "integrationMode": "foundry_agent365_governance",
  "azure": {
    "subscriptionId": "c200e3e7-0839-483b-848b-25a98451f2cd",
    "resourceGroup": "rg-kotp-temp"
  },
  "foundry": {
    "projectEndpoint": "https://aif-aiagents-vbds.services.ai.azure.com/api/projects/workshop-project",
    "projectName": "workshop-project",
    "accountName": "aif-aiagents-vbds",
    "accountKind": "AIServices",
    "accountLocation": "swedencentral",
    "accountResourceId": "/subscriptions/c200e3e7-0839-483b-848b-25a98451f2cd/resourceGroups/rg-kotp-temp/providers/Microsoft.CognitiveServices/accounts/aif-aiagents-vbds",
    "modelDeploymentName": "gp

## Step 6 — Verify the Foundry → Agent 365 governance path

This verification checks the local evidence and the Azure-side Agent 365 configuration. The final registry confirmation happens in the Microsoft 365 admin center.


In [8]:
with open(evidence_json_path, "r", encoding="utf-8") as f:
    evidence_check = json.load(f)

print("[VERIFY] Integration mode:", evidence_check["integrationMode"])
print("[VERIFY] Foundry agent:", evidence_check["foundry"]["agentName"])
print("[VERIFY] Foundry resource:", evidence_check["foundry"]["accountResourceId"])
print("[VERIFY] a365LoggingEnabled:", evidence_check["agent365"]["a365LoggingEnabled"])
print("[VERIFY] a365Status:", evidence_check["agent365"].get("a365Status"))

assert evidence_check["integrationMode"] == "foundry_agent365_governance"
assert evidence_check["foundry"]["agentName"] == FOUNDRY_AGENT_NAME
assert evidence_check["foundry"]["projectEndpoint"] == PROJECT_ENDPOINT
assert evidence_check["agent365"]["a365LoggingEnabled"] is True

fresh_account = read_foundry_account(FOUNDRY_ACCOUNT_RESOURCE_ID)
fresh_props = fresh_account.get("properties", {})
assert fresh_props.get("a365LoggingEnabled") is True

print("[SUCCESS] Foundry resource and local evidence are ready for Agent 365 governance validation.")


[VERIFY] Integration mode: foundry_agent365_governance
[VERIFY] Foundry agent: foundry-lab-agent-vscode-dd3ae0
[VERIFY] Foundry resource: /subscriptions/c200e3e7-0839-483b-848b-25a98451f2cd/resourceGroups/rg-kotp-temp/providers/Microsoft.CognitiveServices/accounts/aif-aiagents-vbds
[VERIFY] a365LoggingEnabled: True
[VERIFY] a365Status: Enabled
[SUCCESS] Foundry resource and local evidence are ready for Agent 365 governance validation.


## Step 7 — Confirm in Microsoft 365 admin center

After the Foundry agent and data collection setting are in place:

1. Open `https://admin.microsoft.com`.
2. Go to **Agents**.
3. Search for the Foundry agent name, for example `Foundry Lab Agent` or the generated agent name printed by this notebook.
4. Open the details pane and note:
   - Agent name
   - Agent source / origin
   - Agent identity
   - Tools or capabilities
   - Availability
   - User/group assignment
   - Status
   - Policy or restriction state

For the governance demo, use the admin center to show:

| Demo | What to show |
|------|--------------|
| Discovery | The Foundry agent appears in the Agent inventory |
| Availability | Restrict availability to selected users or groups |
| Disable/enable | Block and unblock the agent |
| Lifecycle | Show the agent as a managed inventory item |
| Metadata | Show source, identity, tools, and other visible details |
| Audit/activity | Invoke the agent, then inspect the available activity or audit surfaces |


## Step 8 — Policy demo checklist

Use this section as the script for your governance demo.

| Policy area | Demo action | Expected effect |
|-------------|-------------|-----------------|
| Access policy | Limit availability to a test group | Users outside the group cannot use or discover the agent as allowed by tenant policy |
| Block policy | Disable the agent | Agent becomes unavailable according to Agent 365 policy state |
| Conditional Access | Apply tenant access rules to users or managed agent paths | Access follows tenant security rules |
| Data boundary | Test a prompt that would trigger data handling expectations | Tenant and Foundry controls determine the allowed behavior |
| Audit | Invoke the Foundry agent, then inspect audit or activity surfaces | Activity should be attributable to the governed Foundry agent path |
| Lifecycle | Update the Foundry agent and inspect inventory state | Admins can see and manage the agent as part of the organizational agent estate |

The key point of the demo is that the agent is created in Foundry, while Agent 365 gives administrators a governance view and policy surface for managing it.


## Step 9 — Optional cleanup

This lab does not delete Foundry or Agent 365 resources by default.

Set `RUN_CLEANUP_LOCAL_ONLY = True` if you only want to remove generated local evidence files.


In [9]:
RUN_CLEANUP_LOCAL_ONLY = os.environ.get("RUN_CLEANUP_LOCAL_ONLY", "false").lower() == "true"

if RUN_CLEANUP_LOCAL_ONLY:
    for file_name in [
        "agent365-governance-evidence.json",
        "agent365-governance-evidence.md",
    ]:
        file_path = LAB_OUTPUT_DIR / file_name
        if file_path.exists():
            file_path.unlink()
            print("[INFO] Deleted:", file_path)
    print("[SUCCESS] Local generated files removed.")
else:
    print("[INFO] RUN_CLEANUP_LOCAL_ONLY is false. No cleanup performed.")


[INFO] RUN_CLEANUP_LOCAL_ONLY is false. No cleanup performed.
